In [1]:
import pandas as pd
import time
import os
import re
import sys
from IPython.display import clear_output
# import gspread

sys.path.append('./src')
from GSheetImporter import GSheetImporter
from pullprice import yfinance_sym_dic, get_live_price

# Load arguments

In [2]:
GENERATE_MARKDOWN = True
GENERATE_HTML = True

REPORT_PATH = '../TradingAssistWebapp/pages/'
# GSHEET_CREDS = "c:/users/pbara/Documents/Python/secrets/sheets-pandas-reader-193e91a08e8e.json"
GSHEET_CREDS = '/home/pbarahimi/.credentials/gsheets.json'

# Read the trades worksheet

- Requires google account service credentials to read the full sheet regardless of the filters applied in the browser
- Refer to [`gsheet_access_instructions.txt`](https://share.gemini.google/0dLETC0zvoAe) for step-by-step instructions on how to setup access

In [3]:
SHEET_ID = "1HJ9h7UEtUQCXNA58UkZyPsHogJWBAcB1lNWt9nOPMR4"
SHEET_NAME = 'Trades'
num_cols = ['Open Price', 'Close Price', 'Commission','Risk ($)', 'Balance at Open', 'PnL']

gsheet = GSheetImporter(sheet_id=SHEET_ID, sheet_name=SHEET_NAME, credentials_path=GSHEET_CREDS)
gsheet.get_dataframe()
gsheet.to_num(num_cols)

# Keep open trades
df = gsheet.df[gsheet.df['Is Closed']==0].copy()

df.head()

,Open Time,Open Date,Account,Symbol,Volume,Open Price,Close Price,Commission,Balance at Open,Stop Loss,...,Closed,Risk ($),Potential Profit,PnL,Is Closed,Close Time,Close Date,Entry Link,Exit Link 1,Strategy
179,9/3/2026 10:11:59,09/03/26,Paper Trading #2,TSLA,-24.0,371.81,364.27,0.00,100608.20,411.98,...,No,-964.08,2444.88,180.96,0,,#VALUE!,"Trade 1, 1h,",,
183,8/24/2026 0:00:00,08/24/26,Tradestation - Equity,CVNA,-140.0,72.32,65.11,-2.91,42336.39,81.13,...,No,-1172.61,2250.29,1007.09,0,,#VALUE!,Link,,
184,9/2/2026 13:35:00,09/02/26,Tradestation - Equity,CVNA,-25.0,74.81,65.11,0.00,48760.00,76.8,...,No,-49.75,170.25,242.50,0,,#VALUE!,,,
198,9/6/2026 10:00:00,06/13/26,Tradestation - Futures,ADA,177000.0,0.17,NaN,0.00,218000.00,,...,No,0.00,500290.5,0.00,0,,#VALUE!,,,
269,8/16/2026 0:00:00,08/16/26,Yvonne's Robinhood,ADA,60000.0,0.18,NaN,0.00,218000.00,,...,No,0.00,169192.71,0.00,0,,#VALUE!,,,


# Get Point Values

In [4]:
# Specify the tab name (optional, defaults to the first sheet)
SHEET_NAME = 'Symbols'

gsheet = GSheetImporter(SHEET_ID, SHEET_NAME, GSHEET_CREDS)
all_values = gsheet.get_all_values()
point_val_df = pd.DataFrame(all_values, columns=['Symbol', 'Point Value'])
point_val_df['Point Value'] = point_val_df['Point Value'].astype(float)

point_val_df.head()

,Symbol,Point Value
0,ADA,1.0
1,BTC,1.0
2,COF,1.0
3,CVNA,1.0
4,ETH,1.0


# Get Prices

In [5]:
price_df = pd.DataFrame(df['Symbol']).drop_duplicates()
price_df['Current Price'] = price_df.Symbol.apply(lambda x : get_live_price(x, yfinance_sym_dic))
price_df

,Symbol,Current Price
179,TSLA,364.269989
183,CVNA,65.110001
198,ADA,0.227140


# Append Price to trades DF

In [6]:
df = pd.merge(df, price_df, on='Symbol', how='left')
df = pd.merge(df, point_val_df, on='Symbol', how='left')
df['Point Value'] = df['Point Value'].fillna(1)
df['PnL'] = (df['Volume'] * (df['Current Price']-df['Open Price']) * df['Point Value']).round(2)
df

,Open Time,Open Date,Account,Symbol,Volume,Open Price,Close Price,Commission,Balance at Open,Stop Loss,...,Potential Profit,PnL,Is Closed,Close Time,Close Date,Entry Link,Exit Link 1,Strategy,Current Price,Point Value
0,9/3/2026 10:11:59,09/03/26,Paper Trading #2,TSLA,-24.0,371.81,364.27,0.00,100608.20,411.98,...,2444.88,180.96,0,,#VALUE!,"Trade 1, 1h,",,,364.269989,1.0
1,8/24/2026 0:00:00,08/24/26,Tradestation - Equity,CVNA,-140.0,72.32,65.11,-2.91,42336.39,81.13,...,2250.29,1009.40,0,,#VALUE!,Link,,,65.110001,1.0
2,9/2/2026 13:35:00,09/02/26,Tradestation - Equity,CVNA,-25.0,74.81,65.11,0.00,48760.00,76.8,...,170.25,242.50,0,,#VALUE!,,,,65.110001,1.0
3,9/6/2026 10:00:00,06/13/26,Tradestation - Futures,ADA,177000.0,0.17,NaN,0.00,218000.00,,...,500290.5,10113.78,0,,#VALUE!,,,,0.227140,1.0
4,8/16/2026 0:00:00,08/16/26,Yvonne's Robinhood,ADA,60000.0,0.18,NaN,0.00,218000.00,,...,169192.71,2828.40,0,,#VALUE!,,,,0.227140,1.0
5,9/9/2026 9:52:13,09/09/26,Tradestation - Equity,TSLA,-4.0,371.98,364.27,0.00,56000.00,385.26,...,396.8,30.84,0,,#VALUE!,#REF!,,Fibo,364.269989,1.0
6,9/3/2026 9:59:40,09/03/26,Paper Trading #1,TSLA,-14.0,371.81,364.27,0.00,61600.00,411.98,...,1426.18,105.56,0,,#VALUE!,"Trade 1, 1h,",,Fibo,364.269989,1.0


# Group by account and symbol to report

In [7]:
spacer_line = '\n\n' + 50 * '-' + '\n'
out = df.groupby('Account').agg({'PnL': sum}).to_string() + spacer_line
out += df.groupby('Symbol').agg({'Volume': sum, 'PnL': sum}).to_string() + spacer_line
out += df.groupby(['Symbol','Account']).agg({'Volume': sum, 'PnL': sum}).to_string() + spacer_line
out += df.groupby(['Account','Symbol']).agg({'Volume': sum, 'PnL': sum}).to_string() + spacer_line
print(out)

                             PnL
Account                         
Paper Trading #1          105.56
Paper Trading #2          180.96
Tradestation - Equity    1282.74
Tradestation - Futures  10113.78
Yvonne's Robinhood       2828.40

--------------------------------------------------
          Volume       PnL
Symbol                    
ADA     237000.0  12942.18
CVNA      -165.0   1251.90
TSLA       -42.0    317.36

--------------------------------------------------
                                 Volume       PnL
Symbol Account                                   
ADA    Tradestation - Futures  177000.0  10113.78
       Yvonne's Robinhood       60000.0   2828.40
CVNA   Tradestation - Equity     -165.0   1251.90
TSLA   Paper Trading #1           -14.0    105.56
       Paper Trading #2           -24.0    180.96
       Tradestation - Equity       -4.0     30.84

--------------------------------------------------
                                 Volume       PnL
Account                Symbo

In [8]:
'''
spacer_line = '\n\n<br>\n\n' 

out = df.groupby('Account').agg({'PnL': sum}).to_markdown() + spacer_line
out += df.groupby('Symbol').agg({'Volume': sum, 'PnL': sum}).to_markdown() + spacer_line
out += df.groupby(['Symbol','Account'], as_index=False).agg({'Volume': sum, 'PnL': sum}).to_markdown() + spacer_line
out += df.groupby(['Account','Symbol'], as_index=False).agg({'Volume': sum, 'PnL': sum}).to_markdown() + spacer_line
'''
out = df.groupby(['Account','Symbol'], as_index=False).agg({'Volume': sum, 'PnL': sum}).to_markdown()
print(out)

|    | Account                | Symbol   |   Volume |      PnL |
|---:|:-----------------------|:---------|---------:|---------:|
|  0 | Paper Trading #1       | TSLA     |      -14 |   105.56 |
|  1 | Paper Trading #2       | TSLA     |      -24 |   180.96 |
|  2 | Tradestation - Equity  | CVNA     |     -165 |  1251.9  |
|  3 | Tradestation - Equity  | TSLA     |       -4 |    30.84 |
|  4 | Tradestation - Futures | ADA      |   177000 | 10113.8  |
|  5 | Yvonne's Robinhood     | ADA      |    60000 |  2828.4  |


In [9]:
if GENERATE_MARKDOWN:
    '''
    page_nm = 'acct_lvl_stats.md'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:  # Save to a file
        f.write(df.groupby('Account').agg({'PnL': sum}).to_markdown())
        
    page_nm = 'sym_lvl_stats.md'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        f.write(df.groupby('Symbol').agg({'Volume': sum, 'PnL': sum}).to_markdown())
    
    page_nm = 'sym_acct_lvl_stats.md'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        f.write(df.groupby(['Symbol','Account'], as_index=False).agg({'Volume': sum, 'PnL': sum}).to_markdown())
    '''
    page_nm = 'acct_sym_lvl_stats.md'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        f.write(df.groupby(['Account','Symbol'], as_index=False).agg({'Volume': sum, 'PnL': sum}).to_markdown())

In [10]:
if GENERATE_HTML:
    '''
    page_nm = 'acct_lvl_stats.html'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:  # Save to a file
        t = df.groupby('Account').agg({'PnL': sum})
        f.write(t.to_html(border=0, justify='left',  table_id='dataTable', classes='table table-striped table-hover'))
        
    page_nm = 'sym_lvl_stats.html'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        t = df.groupby('Symbol').agg({'Volume': sum, 'PnL': sum})
        f.write(t.to_html(border=0, justify='left',  table_id='dataTable', classes='table table-striped table-hover'))
    
    page_nm = 'sym_acct_lvl_stats.html'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        t = df.groupby(['Symbol','Account']).agg({'Volume': sum, 'PnL': sum})
        f.write(t.to_html(border=0, justify='left',  table_id='dataTable', classes='table table-striped table-hover'))
    '''
    page_nm = 'acct_sym_lvl_stats.html'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        t = df.groupby(['Account','Symbol']).agg({'Volume': sum, 'PnL': sum})
        f.write(t.to_html(border=0, justify='left',  table_id='dataTable', classes='table table-striped table-hover'))